# Individual Differences Integration

Per-participant measures from across the analysis notebooks, correlated to test whether gaze-cursor lag, TTI, regression rate, and LHIPA form a coherent processing speed / search strategy dimension.

**DVs per participant:**
- Gaze-cursor lag (from `gaze_cursor_lag.ipynb`) — median ms, negative = gaze leads
- TTI to first scroll — time from page load to first scroll event
- Regression rate — fraction of trials with ≥1 backward scroll
- Mean click position — average rank of clicked result
- LHIPA — mean Low/High Index of Pupillary Activity (cognitive load, lower = more load)
- Fixation count per trial — mean across trials
- Trial duration — mean seconds

In [ ]:
from data_loader import *
setup_plotting()

import numpy as np
from scipy import stats
from collections import defaultdict
import matplotlib.pyplot as plt


> *Functions ,  moved to .*

In [ ]:
# ── Compute all per-trial measures ──────────────────────────────

# Load pre-computed LHIPA (dict keyed by trial_id)
lhipa_data = {}
lhipa_path = Path('lhipa_per_trial.json')
if lhipa_path.exists():
    with open(lhipa_path) as f:
        raw = json.load(f)
        for tid, entry in raw.items():
            lhipa_data[tid] = entry['lhipa']
    print(f'LHIPA loaded: {len(lhipa_data)} trials')
else:
    print('No LHIPA data — run scripts/compute_lhipa.py first')

trial_ids = sorted(p.stem for p in FIX_DIR.glob('*.csv'))
trial_measures = []

for i, tid in enumerate(trial_ids):
    if i % 500 == 0: print(f'  Processing {i}/{len(trial_ids)}...')
    try:
        fix = load_fixations(tid)
        mouse, scrolls, clicks = load_mouse_and_scroll(tid)
        if len(fix) < 3 or len(mouse) < 5: continue
        
        pid = tid.split('-')[0]
        t_start = min(fix[0][0], mouse[0][0])
        t_end = max(fix[-1][0] + fix[-1][3], mouse[-1][0])
        duration_s = (t_end - t_start) / 1000
        
        # TTI to first scroll
        tti_scroll = (scrolls[0][0] - t_start) / 1000 if scrolls else None
        
        # Orientation: time to first fixation
        orientation_ms = fix[0][0] - t_start
        
        # Regression
        regressed = has_regression(scrolls)
        
        # Gaze-cursor lag
        lag = compute_lag_for_trial(fix, mouse, scrolls)
        
        # Click position (rough: click Y / ~150px per result)
        click_pos = None
        if clicks:
            click_y = clicks[-1][2]  # last click, page-space Y
            click_pos = click_y / 150  # rough position estimate
        
        # LHIPA
        lhipa = lhipa_data.get(tid)
        
        trial_measures.append({
            'trial_id': tid,
            'participant': pid,
            'n_fixations': len(fix),
            'duration_s': duration_s,
            'tti_scroll_s': tti_scroll,
            'orientation_ms': orientation_ms,
            'regressed': regressed,
            'lag_ms': lag,
            'click_pos': click_pos,
            'lhipa': lhipa,
        })
    except Exception:
        pass

print(f'\nProcessed {len(trial_measures)} trials')

In [ ]:
# ── Aggregate per participant ───────────────────────────────────

by_pid = defaultdict(list)
for m in trial_measures:
    by_pid[m['participant']].append(m)

participant_dvs = {}
for pid, trials in by_pid.items():
    lags = [t['lag_ms'] for t in trials if t['lag_ms'] is not None]
    ttis = [t['tti_scroll_s'] for t in trials if t['tti_scroll_s'] is not None]
    lhipas = [t['lhipa'] for t in trials if t['lhipa'] is not None]
    click_positions = [t['click_pos'] for t in trials if t['click_pos'] is not None]
    
    participant_dvs[pid] = {
        'n_trials': len(trials),
        'median_lag_ms': np.median(lags) if len(lags) >= 5 else None,
        'lag_sd': np.std(lags) if len(lags) >= 5 else None,
        'median_tti_s': np.median(ttis) if len(ttis) >= 5 else None,
        'regression_rate': np.mean([t['regressed'] for t in trials]),
        'mean_fixations': np.mean([t['n_fixations'] for t in trials]),
        'mean_duration_s': np.mean([t['duration_s'] for t in trials]),
        'mean_click_pos': np.mean(click_positions) if click_positions else None,
        'mean_lhipa': np.mean(lhipas) if len(lhipas) >= 5 else None,
        'mean_orientation_ms': np.median([t['orientation_ms'] for t in trials]),
    }

print(f'Participants with all measures:')
complete = {pid: dv for pid, dv in participant_dvs.items() 
            if all(dv[k] is not None for k in ['median_lag_ms','median_tti_s','mean_lhipa','mean_click_pos'])}
print(f'  {len(complete)} / {len(participant_dvs)}')

# Quick stats
for key in ['median_lag_ms', 'median_tti_s', 'regression_rate', 'mean_lhipa', 'mean_click_pos', 'mean_fixations']:
    vals = [dv[key] for dv in complete.values() if dv[key] is not None]
    print(f'  {key}: median={np.median(vals):.1f}, SD={np.std(vals):.1f}, range={np.min(vals):.1f}–{np.max(vals):.1f}')

In [ ]:
# ── Correlation matrix ──────────────────────────────────────────

keys = ['median_lag_ms', 'median_tti_s', 'regression_rate', 'mean_lhipa', 'mean_click_pos', 'mean_fixations', 'mean_duration_s']
labels = ['Gaze-cursor\nlag (ms)', 'TTI to\nscroll (s)', 'Regression\nrate', 'LHIPA\n(cog load)', 'Click\nposition', 'Fixation\ncount', 'Duration\n(s)']

# Build matrix from complete participants
pids = sorted(complete.keys())
matrix = np.array([[complete[pid][k] for k in keys] for pid in pids])

n = len(keys)
rho_matrix = np.zeros((n, n))
p_matrix = np.ones((n, n))

print(f'Spearman correlations (n={len(pids)} participants):\n')
print(f'{"":>20s}', end='')
for l in labels:
    print(f'{l.split(chr(10))[0]:>12s}', end='')
print()

for i in range(n):
    print(f'{labels[i].split(chr(10))[0]:>20s}', end='')
    for j in range(n):
        valid = np.isfinite(matrix[:, i]) & np.isfinite(matrix[:, j])
        if valid.sum() >= 10:
            rho, p = stats.spearmanr(matrix[valid, i], matrix[valid, j])
            rho_matrix[i, j] = rho
            p_matrix[i, j] = p
            sig = '*' if p < 0.05 else ' '
            print(f'{rho:>+11.2f}{sig}', end='')
        else:
            print(f'{"—":>12s}', end='')
    print()

print(f'\n* p < 0.05')

In [ ]:
# ── Key hypothesis tests ────────────────────────────────────────

def report_corr(label, x_key, y_key):
    x = np.array([complete[pid][x_key] for pid in pids])
    y = np.array([complete[pid][y_key] for pid in pids])
    valid = np.isfinite(x) & np.isfinite(y)
    rho, p = stats.spearmanr(x[valid], y[valid])
    print(f'{label}: ρ = {rho:+.3f}, p = {p:.4f}, n = {valid.sum()}')
    return x[valid], y[valid], rho, p

print('=== Key hypotheses ===\n')

print('H1: Gaze-cursor lag correlates with TTI (both measure processing speed)')
x1, y1, r1, p1 = report_corr('  Lag × TTI', 'median_lag_ms', 'median_tti_s')

print('\nH2: Gaze-cursor lag correlates with regression rate (strategy dimension)')
x2, y2, r2, p2 = report_corr('  Lag × Regression rate', 'median_lag_ms', 'regression_rate')

print('\nH3: Gaze-cursor lag correlates with LHIPA (cognitive load)')
x3, y3, r3, p3 = report_corr('  Lag × LHIPA', 'median_lag_ms', 'mean_lhipa')

print('\nH4: Regression rate correlates with LHIPA (replication of our earlier finding)')
x4, y4, r4, p4 = report_corr('  Regression rate × LHIPA', 'regression_rate', 'mean_lhipa')

print('\nH5: TTI correlates with regression rate')
x5, y5, r5, p5 = report_corr('  TTI × Regression rate', 'median_tti_s', 'regression_rate')

print('\nH6: Click position correlates with LHIPA (deeper foraging = more load)')
x6, y6, r6, p6 = report_corr('  Click position × LHIPA', 'mean_click_pos', 'mean_lhipa')

In [ ]:
# ── Scatter plots for significant correlations ──────────────────

pairs = [
    ('median_lag_ms', 'median_tti_s', 'Gaze-cursor lag (ms)', 'TTI to scroll (s)', r1, p1),
    ('median_lag_ms', 'regression_rate', 'Gaze-cursor lag (ms)', 'Regression rate', r2, p2),
    ('median_lag_ms', 'mean_lhipa', 'Gaze-cursor lag (ms)', 'LHIPA (↓ = more load)', r3, p3),
    ('regression_rate', 'mean_lhipa', 'Regression rate', 'LHIPA (↓ = more load)', r4, p4),
    ('median_tti_s', 'regression_rate', 'TTI to scroll (s)', 'Regression rate', r5, p5),
    ('mean_click_pos', 'mean_lhipa', 'Click position', 'LHIPA', r6, p6),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for i, (xk, yk, xl, yl, rho, p) in enumerate(pairs):
    ax = axes[i]
    x = np.array([complete[pid][xk] for pid in pids])
    y = np.array([complete[pid][yk] for pid in pids])
    valid = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[valid], y[valid], alpha=0.6, s=30, color='steelblue')
    
    # Trend line
    if valid.sum() >= 5:
        z = np.polyfit(x[valid], y[valid], 1)
        xline = np.linspace(np.min(x[valid]), np.max(x[valid]), 50)
        ax.plot(xline, np.polyval(z, xline), 'r-', alpha=0.5)
    
    sig = '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.set_title(f'ρ = {rho:+.2f} ({sig})', fontsize=10)

plt.tight_layout()
plt.savefig('plot_individual_differences.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_individual_differences.png')

In [ ]:
# ── Within-participant stability (ICC proxy) ───────────────────
# For gaze-cursor lag: is it a stable trait or state-dependent?

# Split each participant's trials into odd/even halves
odd_medians, even_medians = [], []
for pid in pids:
    trials_with_lag = [t for t in by_pid[pid] if t['lag_ms'] is not None]
    if len(trials_with_lag) < 10: continue
    odd = [t['lag_ms'] for i, t in enumerate(trials_with_lag) if i % 2 == 1]
    even = [t['lag_ms'] for i, t in enumerate(trials_with_lag) if i % 2 == 0]
    odd_medians.append(np.median(odd))
    even_medians.append(np.median(even))

odd_medians = np.array(odd_medians)
even_medians = np.array(even_medians)

if len(odd_medians) >= 10:
    split_half_r, split_half_p = stats.pearsonr(odd_medians, even_medians)
    print(f'Split-half reliability of gaze-cursor lag:')
    print(f'  r = {split_half_r:.3f}, p = {split_half_p:.4f}, n = {len(odd_medians)} participants')
    print(f'  Spearman-Brown corrected: {2*split_half_r/(1+split_half_r):.3f}')
    print(f'  (> 0.7 = acceptable reliability as individual difference measure)')
    
    plt.figure(figsize=(5, 5))
    plt.scatter(odd_medians, even_medians, alpha=0.6, s=30)
    lims = [min(odd_medians.min(), even_medians.min()), max(odd_medians.max(), even_medians.max())]
    plt.plot(lims, lims, 'k--', alpha=0.3)
    plt.xlabel('Odd-trial median lag (ms)')
    plt.ylabel('Even-trial median lag (ms)')
    plt.title(f'Split-half reliability: r = {split_half_r:.2f}')
    plt.tight_layout()
    plt.savefig('plot_lag_split_half.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Not enough participants with ≥10 trials for split-half: {len(odd_medians)}')

## Summary

If the key correlations are significant, it suggests a coherent individual-differences dimension:

**Processing speed / deliberation trait:**
- Faster gaze-cursor lag → faster TTI → lower regression rate → shallower click → lower cognitive load
- = "satisficer" profile

**Slower / more deliberate:**
- Longer lag → slower TTI → more regressions → deeper (or higher) click → more cognitive load
- = "optimizer" profile

The gaze-cursor lag would be a third independent measure of this dimension, alongside TTI and regression rate — converging evidence from motor, temporal, and behavioral signals.

### Key Measures

| Measure | Definition | Units | Interpretation |
|---------|-----------|-------|----------------|
| Gaze-cursor lag | Per-participant median temporal offset (scroll-corrected cross-correlation) | ms | Motor timing dimension; negative = gaze leads |
| TTI to first scroll | Per-participant median time from page load to first scroll | ms | Temporal behavior dimension; proxy for processing speed |
| Regression rate | Per-participant fraction of trials with scroll regressions | % | Behavioral strategy dimension; satisfice vs optimize |
| Mean LHIPA | Per-participant average cognitive load index | ratio | Cognitive load dimension; lower = more effortful processing |
| Mean click position | Per-participant average rank of clicked result | rank | Outcome measure; deeper = more exploration |
